In [28]:
from Transformer import GroupQueryAttentionKVCache
import torch 
from torch import nn
import numpy as np

In [29]:
QWEN_CONFIG_06_B = {
    "vocab_size": 151_936,     # Vocabulary size
    "context_length": 40_960,  # Length originally used during training
    "emb_dim": 1024,           # Embedding dimension
    "n_heads": 16,             # Number of attention heads
    "n_layers": 28,            # Number of layers
    "hidden_dim": 3072,        # Size of intermediate dim in FeedForward
    "head_dim": 128,           # Size of the heads in GQA
    "qk_norm": True,           # Whether to normalize queries & keys in GQA
    "n_kv_groups": 8,          # Key-Value groups for GQA
    "rope_base": 1_000_000.0,  # The base in RoPE's "theta"
    "dtype": torch.bfloat16,   # Lower-precision dtype to reduce memory
}

# Feed forward layer

> Time complexity: O(seq_len * d_model^2), if d_hidden = 4 * d_model.

> Memory complexity: O(seq_len * d_model)

In [30]:
class FeedForward(nn.Module):
    def __init__(self, d_in, d_hidden):
        super().__init__()
        self.d_in = d_in
        self.d_hidden = d_hidden
        self.Linear1 = nn.Linear(d_in, d_hidden)
        self.Linear2 = nn.Linear(d_hidden, d_in)

    def forward(self, x):
        x = self.Linear1(x)
        x = nn.ReLU()(x)
        x = self.Linear2(x)
        return x

In [31]:
batch, sentence_length, embedding_dim = 20, 5, 10
embedding = torch.randn(batch, sentence_length, embedding_dim)
layer_norm = nn.LayerNorm(embedding_dim)
# Activate module
layer_norm(embedding).sum(dim=-1).shape

torch.Size([20, 5])

# Normalization layer

In [32]:
class LayerNormalization(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
        self.bias = nn.Parameter(torch.zeros(dim))

    def forward(self, x):
        me, var = x.mean(dim=-1, keepdim=True), x.var(dim=-1, keepdim=True, unbiased=False)
        x_norm = (x - me)/(torch.sqrt(var+self.eps))
        return x_norm * self.weight + self.bias
        

In [33]:
x = torch.rand(8,5,16)
ln = LayerNormalization(16)
ln(x).shape

torch.Size([8, 5, 16])

> In Qwen3, we use RMSNorm because it is much more memory efficient and computationally efficient.
> 
> LayerNorm has time complexity O(seq_len * d_model), and memory complexity O(seq_len * d_model)
>
> RMSNorm is less sensitive to outliers since it doesn’t subtract the mean

In [34]:
class RMSNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-6, bias=False, qwen3_compatible=True):
        super().__init__()
        self.eps = eps
        self.qwen3_compatible = qwen3_compatible
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim)) if bias else None

    def forward(self, x):
        input_dtype = x.dtype

        if self.qwen3_compatible:
            x = x.to(torch.float32)

        var = x.pow(2).mean(dim=-1, keepdim=True)
        norm_x = x * torch.rsqrt(var + self.eps)
        norm_x = norm_x * self.scale

        if self.shift is not None:
            norm_x = norm_x + self.shift

        return norm_x.to(input_dtype)

In [35]:
x = torch.rand(8,5,16)
ln = RMSNorm(16)
ln(x).shape

torch.Size([8, 5, 16])

# TransformerBlock

In [36]:
class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.d_model = config['emb_dim']
        self.d_hidden = config['hidden_dim']
        self.n_heads = config['n_heads']
        self.context_length = config['context_length']
        self.n_kv_groups = config['n_kv_groups']
        
        self.norm1 = RMSNorm(self.d_model)
        self.norm2 = RMSNorm(self.d_model)
        self.attention_layer = GroupQueryAttentionKVCache(self.d_model, self.n_heads, self.context_length, self.n_kv_groups)
        self.ff_layer = FeedForward(self.d_model, self.d_hidden)

    def forward(self, x, use_cache=False):
        x = x + self.attention_layer(self.norm1(x), use_cache) # Pre-layernorm
        x = x + self.ff_layer(self.norm2(x))
        return x
        

In [37]:
block = TransformerBlock(QWEN_CONFIG_06_B)

In [38]:
bs, seq_len = 8, 15
context_window = 8
x = torch.rand((bs, seq_len, QWEN_CONFIG_06_B['emb_dim']))

In [39]:
block(x).shape

torch.Size([8, 15, 1024])

## Positional Encodings

## Original pe
<img src="positional_encoding.png" width=200 height=100>
i represents the i-th element in the embedding

In [40]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, seq_len: int, dropout: float) -> None:
        super().__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        self.dropout = nn.Dropout(dropout)
        
        pe = torch.zeros(seq_len, d_model) # initialize the size of positional encoder
        pos = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1) # has shape (seq_len, 1)
        denom = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)) # denominator, with size d_model/2
        pe[:,::2] = torch.sin(pos*denom)
        pe[:,1::2] = torch.cos(pos*denom)
        pe = pe.unsqueeze(0) # (1, seq_len, d_model)
        
        self.register_buffer('pe',pe)
        
    def forward(self, x):
        x = x + (self.pe[:, :x.shape[1], :]).requires_grad_(False)
        return self.dropout(x)


Most LLMs now use [ROPE](https://arxiv.org/pdf/2104.09864) because:
> it captures relative position rather than absolute position
>
> Strong extrapolation ability: Instead of adding a vector, RoPE rotates query and key vectors by an angle dependent on their position in the sequence
>
> Efficient and parameter-free

In [52]:
class RoPE(nn.Module):
    def __init__(self, d_model, base = 10000.0):
        super().__init__()
        assert d_model % 2 == 0, "Error: d_model must be divisible by 2"
        self.d_model = d_model
        self.theta = torch.tensor([np.exp(-2*(i//2)*np.log(base)/self.d_model) for i in range(self.d_model)])

    def forward(self, x):
        seq_len = x.shape[1]
        x_reshaped = x.view(x.shape[0], x.shape[1], self.d_model//2, 2)
        x_new = (x_reshaped @ torch.tensor([[0,1],[-1,0]], dtype=x.dtype)).view(x.shape[0], x.shape[1], x.shape[2])
        
        cos = torch.cos(torch.arange(1,seq_len+1).unsqueeze(1) * self.theta.unsqueeze(0)) # (seq_len * d_model)
        sin = torch.sin(torch.arange(1,seq_len+1).unsqueeze(1) * self.theta.unsqueeze(0))

        return x * cos + x_new * sin

In [53]:
pe = RoPE(16)

In [55]:
x = torch.rand((4,7,16))

In [57]:
pe(x).shape

torch.Size([4, 7, 16])